# 🤖 Agent Development - Risk Analyst & Compliance Officer

Welcome to Phase 2 & 3 of the project! In this notebook, you'll develop and test both AI agents:

1. **Risk Analyst Agent** (Chain-of-Thought prompting)
2. **Compliance Officer Agent** (ReACT prompting)

## 🎯 Learning Objectives
- Implement Chain-of-Thought prompting for systematic reasoning
- Build ReACT prompting for structured action-taking
- Handle structured JSON outputs from LLMs
- Create robust error handling and validation
- Test agents with real financial data

## 🚀 Setup and Environment

In [1]:
# Import required libraries
import os
import sys
import json
import openai
from dotenv import load_dotenv
from datetime import datetime

# Add project root + src so `src` package is importable
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, ".."))
src_path = os.path.join(project_root, "src")
for path in [project_root, src_path]:
    if path not in sys.path:
        sys.path.insert(0, path)

# Load environment variables
load_dotenv(os.path.join(project_root, ".env"))

print("📚 Libraries loaded!")
print("🔐 Environment variables loaded")
print("📂 Project root on path:", project_root)
print("📂 Src path on path:", src_path)

📚 Libraries loaded!
🔐 Environment variables loaded
📂 Project root on path: /Users/tarun/Documents/Tarun/Learning/Fintech & AI/Udacity/projects/agentic-ai-financial-compliance-sar
📂 Src path on path: /Users/tarun/Documents/Tarun/Learning/Fintech & AI/Udacity/projects/agentic-ai-financial-compliance-sar/src


In [2]:
# OpenAI Setup for Vocareum
import openai

# Option 1: Use the helper function from src package (recommended)
# Uncomment this after implementing foundation_sar.py:
# from src import create_vocareum_openai_client
# client = create_vocareum_openai_client()

# Option 2: Manual setup (for early development)
openai_api_key = os.getenv('OPENAI_API_KEY')

if not openai_api_key:
    print("⚠️ WARNING: No OpenAI API key found!")
    print("Please set OPENAI_API_KEY in your .env file")
    print("Get your Vocareum OpenAI API key from 'Cloud Resources' in your workspace")
else:
    # Vocareum requires routing through their servers
    client = openai.OpenAI(
        base_url="https://openai.vocareum.com/v1",
        api_key=openai_api_key
    )
    print("✅ OpenAI client initialized with Vocareum routing")
    print(f"🔑 API key: {openai_api_key[:8]}...{openai_api_key[-4:]}")
    print("📍 Base URL: https://openai.vocareum.com/v1")
    print("\n💡 Tip: After implementing foundation_sar.py, you can use:")
    print("   from src import create_vocareum_openai_client")
    print("   client = create_vocareum_openai_client()")

✅ OpenAI client initialized with Vocareum routing
🔑 API key: voc-2041...1296
📍 Base URL: https://openai.vocareum.com/v1

💡 Tip: After implementing foundation_sar.py, you can use:
   from src import create_vocareum_openai_client
   client = create_vocareum_openai_client()


## 📊 Phase 1 Review: Load Foundation Components

Before building agents, let's ensure your foundation components are working.

In [3]:
# Import your implemented foundation components
from src.foundation_sar import (
    CustomerData,
    AccountData,
    TransactionData,
    CaseData,
    RiskAnalystOutput,
    ComplianceOfficerOutput,
    ExplainabilityLogger,
    DataLoader,
    load_csv_data,
    audit_session_timestamp_utc,
    audit_jsonl_path,
)

OUTPUTS_DIR = os.path.join(project_root, "outputs")
NOTEBOOK_AGENT_DEV_AUDIT_TS = audit_session_timestamp_utc()

print("✅ Foundation components imported")
print("✅ You can create sample cases for agent testing")

✅ Foundation components imported
✅ You can create sample cases for agent testing


In [4]:
# Test foundation components
# 1. Load CSV data
# 2. Create a DataLoader instance
# 3. Generate a sample case for testing agents

logger = ExplainabilityLogger(
    audit_jsonl_path(
        OUTPUTS_DIR,
        stem="agent_development",
        session_timestamp_utc=NOTEBOOK_AGENT_DEV_AUDIT_TS,
    )
)
loader = DataLoader(logger)

customers_df, accounts_df, transactions_df = load_csv_data("../data/")

customers_df["ssn_last_4"] = customers_df["ssn_last_4"].astype(str)

# Normalize optional fields to strings or None
transactions_df["counterparty"] = transactions_df["counterparty"].where(
    transactions_df["counterparty"].notna(), None
)
transactions_df["location"] = transactions_df["location"].where(
    transactions_df["location"].notna(), None
)

customers_df = customers_df.where(customers_df.notna(), None)
accounts_df = accounts_df.where(accounts_df.notna(), None)
transactions_df = transactions_df.where(transactions_df.notna(), None)

customers_data = customers_df.to_dict("records")
accounts_data = accounts_df.to_dict("records")
transactions_data = transactions_df.to_dict("records")

if not customers_data:
    raise ValueError("No customers loaded from CSV data")

sample_customer = None
for candidate in customers_data:
    candidate_accounts = [
        acc for acc in accounts_data if acc.get("customer_id") == candidate.get("customer_id")
    ]
    candidate_account_ids = {acc.get("account_id") for acc in candidate_accounts}
    candidate_transactions = [
        txn for txn in transactions_data if txn.get("account_id") in candidate_account_ids
    ]
    if candidate_transactions:
        sample_customer = candidate
        break

if not sample_customer:
    raise ValueError("No customer with transactions found in CSV data")

case_data = loader.create_case_from_data(
    sample_customer,
    accounts_data,
    transactions_data,
)

print("✅ Sample case created for agent testing")
print(f"Case ID: {case_data.case_id}")
print(f"Customer: {case_data.customer.name} ({case_data.customer.customer_id})")
print(f"Accounts: {len(case_data.accounts)} | Transactions: {len(case_data.transactions)}")

✅ Sample case created for agent testing
Case ID: e4a55c64-61be-4941-823e-088db73944e6
Customer: Renee Blair (CUST_0002)
Accounts: 1 | Transactions: 11


## 🔍 Phase 2: Risk Analyst Agent Development

The Risk Analyst Agent uses **Chain-of-Thought prompting** to systematically analyze suspicious activity patterns.

### 📚 Understanding Chain-of-Thought Prompting

Chain-of-Thought (CoT) prompting guides AI models through step-by-step reasoning:

1. **Explicit Steps**: Break complex reasoning into clear phases
2. **Sequential Logic**: Each step builds on previous ones
3. **Domain Expertise**: Frame AI as subject matter expert
4. **Structured Output**: Guide toward specific response format

In [5]:
# Chain-of-Thought prompt design (Risk Analyst)
# Aligns with src/risk_analyst_agent.py and RiskAnalystOutput in foundation_sar.py

from src.foundation_sar import risk_analyst_calibration_rubric


def design_cot_prompt():
    """Design Chain-of-Thought system prompt for suspicious-activity risk analysis."""

    system_prompt = """You are a senior financial crime Risk Analyst.
Use Chain-of-Thought reasoning: think step-by-step inside your reasoning field, following the framework below.

Analysis framework (apply in order):
1. Data Review — Summarize customer profile, accounts, and transaction activity relevant to suspicion.
2. Pattern Recognition — List concrete behavioral or transactional indicators (velocity, structuring-like deposits, geography, counterparties, etc.).
3. Regulatory Mapping — Relate indicators to known AML/BSA typologies where applicable (without inventing facts not in the case).
4. Risk Quantification — Explain severity and uncertainty; justify a confidence score between 0.0 and 1.0.
5. Classification Decision — Choose exactly one category that best fits the evidence.

Classification categories (must be one of these strings exactly):
- Structuring
- Sanctions
- Fraud
- Money_Laundering
- Other

Risk level (must be one of): Low, Medium, High, Critical.

Output rules:
- Return ONLY valid JSON, no markdown fences or surrounding prose.
- "reasoning_steps" must be an array of EXACTLY five non-empty strings (steps 1–5 above, in order). Each string ≤400 characters.
- "key_indicators" must be a non-empty list of short strings.
- "confidence_score" must be a number from 0.0 to 1.0.

Required JSON schema:
{
  "classification": "Structuring|Sanctions|Fraud|Money_Laundering|Other",
  "confidence_score": 0.0,
  "reasoning_steps": ["step1 string", "step2 string", "step3 string", "step4 string", "step5 string"],
  "key_indicators": ["indicator1", "indicator2"],
  "risk_level": "Low|Medium|High|Critical"
}

Use professional, regulator-appropriate language. Do not include PII beyond what is already in the case."""

    return (system_prompt + "\n\n" + risk_analyst_calibration_rubric()).strip()


# Inspect prompt design
cot_prompt = design_cot_prompt()
print("🧠 Chain-of-Thought Prompt Design")
print(f"Length: {len(cot_prompt)} characters\n")
print(cot_prompt)


🧠 Chain-of-Thought Prompt Design
Length: 2230 characters

You are a senior financial crime Risk Analyst.
Use Chain-of-Thought reasoning: think step-by-step inside your reasoning field, following the framework below.

Analysis framework (apply in order):
1. Data Review — Summarize customer profile, accounts, and transaction activity relevant to suspicion.
2. Pattern Recognition — List concrete behavioral or transactional indicators (velocity, structuring-like deposits, geography, counterparties, etc.).
3. Regulatory Mapping — Relate indicators to known AML/BSA typologies where applicable (without inventing facts not in the case).
4. Risk Quantification — Explain severity and uncertainty; justify a confidence score between 0.0 and 1.0.
5. Classification Decision — Choose exactly one category that best fits the evidence.

Classification categories (must be one of these strings exactly):
- Structuring
- Sanctions
- Fraud
- Money_Laundering
- Other

Risk level (must be one of): Low, Medium,

In [6]:
# Implement and test Risk Analyst Agent - SIMPLE SMOKE TEST

from src.risk_analyst_agent import RiskAnalystAgent
from src.foundation_sar import ExplainabilityLogger, audit_jsonl_path


def simple_risk_analyst_smoke_test():
    """
    Basic smoke test for the Risk Analyst Agent.
    """
    print("🔍 Risk Analyst Smoke Test")

    if "case_data" not in globals():
        raise ValueError("Run the sample case cell first to define case_data")

    logger = ExplainabilityLogger(
        audit_jsonl_path(
            OUTPUTS_DIR,
            stem="agent_smoke_test",
            session_timestamp_utc=NOTEBOOK_AGENT_DEV_AUDIT_TS,
        )
    )
    agent = RiskAnalystAgent(client, logger)

    try:
        result = agent.analyze_case(case_data)
        print(
            "✅ SUCCESS: Got classification '{}' with confidence {:.2f}".format(
                result.classification, result.confidence_score
            )
        )
    except Exception as exc:
        print(f"❌ FAILED: {exc}")


simple_risk_analyst_smoke_test()

🔍 Risk Analyst Smoke Test
✅ SUCCESS: Got classification 'Other' with confidence 0.45


### 🧪 Risk Analyst Testing Framework

In [7]:
# COMPREHENSIVE Risk Analyst Testing - Import Pre-Built Test Suite
# Uses the provided pytest suite for RiskAnalystAgent

import sys
import os
import subprocess

# Add tests directory to Python path for importing test modules
project_root = os.path.abspath('..')
tests_path = os.path.join(project_root, 'tests')
if tests_path not in sys.path:
    sys.path.insert(0, tests_path)

print(f"📁 Added tests directory to Python path: {tests_path}")

def run_comprehensive_risk_analyst_tests():
    """
    Use pre-built comprehensive test suite to validate your Risk Analyst Agent
    
    These tests validate:
    - Agent initialization and configuration
    - Case analysis with valid inputs
    - JSON parsing and error handling
    - System prompt structure and content
    - API call parameters and responses
    - Helper method functionality
    """
    print("🧪 Comprehensive Risk Analyst Testing")
    print("🚀 Running Risk Analyst test suite...")

    result = subprocess.run(
        ["python", "-m", "pytest", f"{tests_path}/test_risk_analyst.py", "-v", "--tb=short"],
        check=False,
    )

    if result.returncode == 0:
        print("✅ All Risk Analyst tests passed!")
    else:
        print("❌ Some tests failed. Check the output above for details.")

# Quick preview of available tests
try:
    from test_risk_analyst import TestRiskAnalystAgent
    import inspect
    
    # Get all test methods
    test_methods = [method for method in dir(TestRiskAnalystAgent) 
                   if method.startswith('test_')]
    
    print("📊 Preview of Comprehensive Risk Analyst Tests:")
    for method_name in test_methods[:5]:  # Show first 5
        method = getattr(TestRiskAnalystAgent, method_name)
        doc = method.__doc__ or method_name.replace('_', ' ').title()
        print(f"   • {doc}")
    if len(test_methods) > 5:
        print(f"   ... and {len(test_methods) - 5} more tests")
    print("\n💡 These tests validate edge cases you might not think of!")
    print("💡 Much more thorough than manual testing!")
except Exception as e:
    print(f"ℹ️ Test suite will be available after implementing foundation_sar.py: {e}")

# Call the function
run_comprehensive_risk_analyst_tests()

📁 Added tests directory to Python path: /Users/tarun/Documents/Tarun/Learning/Fintech & AI/Udacity/projects/agentic-ai-financial-compliance-sar/tests
📊 Preview of Comprehensive Risk Analyst Tests:
   • Test RiskAnalystAgent initializes properly
   • Full agent path: mocked API returns each Literal label; RiskAnalystOutput validates it.
   • Invalid assistant JSON twice yields conservative fallback; audit log records json_recovery fallback.
   • Malformed first turn; corrective multimessage retry returns valid RiskAnalystOutput.
   • Test Analyze Case Recovers Trailing Comma Without Retry
   ... and 8 more tests

💡 These tests validate edge cases you might not think of!
💡 Much more thorough than manual testing!
🧪 Comprehensive Risk Analyst Testing
🚀 Running Risk Analyst test suite...
============================= test session starts ==============================
platform darwin -- Python 3.12.7, pytest-7.4.4, pluggy-1.0.0 -- /opt/anaconda3/bin/python
cachedir: .pytest_cache
rootdir: /U

## ✅ Phase 3: Compliance Officer Agent Development

The Compliance Officer Agent uses **ReACT prompting** to generate regulatory-compliant SAR narratives.

### 📚 Understanding ReACT Prompting

ReACT (Reasoning + Action) prompting separates thinking and doing:

1. **Reasoning Phase**: Analyze situation and plan approach
2. **Action Phase**: Execute specific task with informed decisions
3. **Structured Workflow**: Consistent approach to complex tasks
4. **Regulatory Compliance**: Emphasis on meeting specific requirements

In [8]:
# TODO: Test ReACT prompt design
# This cell helps you design and test your ReACT prompt structure

def design_react_prompt():
    """Design and test ReACT prompt for compliance narratives"""
    
    system_prompt = """
    TODO: Design your ReACT system prompt here
    
    Key elements to include:
    1. Senior Compliance Officer persona
    2. ReACT framework:
       REASONING Phase:
       - Review risk analyst findings
       - Assess regulatory requirements
       - Identify compliance elements
       - Plan narrative structure
       
       ACTION Phase:
       - Draft concise narrative (≤120 words)
       - Include specific details
       - Reference activity patterns
       - Use regulatory language
    3. JSON output format specification
    """
    
    return system_prompt

# Test your prompt design
react_prompt = design_react_prompt()
print("⚡ ReACT Prompt Design:")
print(react_prompt[:200] + "...")
print("\n📋 TODO: Complete the prompt in compliance_officer_agent.py")

⚡ ReACT Prompt Design:

    TODO: Design your ReACT system prompt here
    
    Key elements to include:
    1. Senior Compliance Officer persona
    2. ReACT framework:
       REASONING Phase:
       - Review risk analyst ...

📋 TODO: Complete the prompt in compliance_officer_agent.py


In [9]:
# Compliance Officer Agent - SIMPLE SMOKE TEST

import os

from src.compliance_officer_agent import ComplianceOfficerAgent
from src.foundation_sar import (
    DataLoader,
    ExplainabilityLogger,
    RiskAnalystOutput,
    load_csv_data,
    audit_session_timestamp_utc,
    audit_jsonl_path,
)

def simple_compliance_officer_smoke_test():
    """
    Basic smoke test to verify the Compliance Officer Agent generates a narrative.
    """
    print("✅ Compliance Officer Smoke Test")

    try:
        if "client" not in globals():
            raise RuntimeError("OpenAI client is not defined. Run the OpenAI setup cell first.")

        # case = load_or_reuse_case()

        _outputs = os.path.join(os.path.abspath(os.path.join(os.getcwd(), "..")), "outputs")
        _logger_path = audit_jsonl_path(
            _outputs,
            stem="compliance_officer_smoke",
            session_timestamp_utc=audit_session_timestamp_utc(),
        )
        logger = ExplainabilityLogger(_logger_path)
        agent = ComplianceOfficerAgent(client, logger)

        test_risk_analysis = RiskAnalystOutput(
            classification="Money_Laundering",
            confidence_score=0.82,
            reasoning_steps=[
                "Data Review: Customer shows elevated balances and wire-heavy activity.",
                "Pattern Recognition: Rapid high-value transfers with offshore counterparties.",
                "Regulatory Mapping: Consistent with layering red flags under BSA review.",
                "Risk Quantification: Confidence 0.82 given velocity and jurisdiction mix.",
                "Classification Decision: Money_Laundering best explains the pattern.",
            ],
            key_indicators=["rapid deposits", "high-value wire transfers"],
            risk_level="High",
        )

        result = agent.generate_compliance_narrative(case_data, test_risk_analysis)
        word_count = len(result.narrative.split()) if result.narrative else 0

        if result.narrative and word_count <= 120:
            print(f"✅ SUCCESS: Generated {word_count} word narrative")
            print(f"Preview: {result.narrative[:120]}...")
        else:
            raise ValueError(
                f"Narrative missing or too long (word_count={word_count})."
            )
    except Exception as exc:
        print(f"❌ FAILED: {exc}")


simple_compliance_officer_smoke_test()


✅ Compliance Officer Smoke Test
❌ FAILED: Compliance narrative failed SAR deterministic validation: missing mandated terminology phrase: "Regulatory threshold"; missing mandated terminology phrase: "Financial institution"; missing mandated terminology phrase: "Bank Secrecy Act"; narrative must echo regulatory citation: "Bank Secrecy Act (BSA)"


### 🧪 Compliance Officer Testing Framework

In [10]:
# COMPREHENSIVE Compliance Officer Testing - Import Pre-Built Test Suite
# Uses the provided pytest suite for ComplianceOfficerAgent

import sys
import os
import subprocess

# Add tests directory to Python path for importing test modules (if not already added)
project_root = os.path.abspath("..")
tests_path = os.path.join(project_root, "tests")
if tests_path not in sys.path:
    sys.path.insert(0, tests_path)

print(f"📁 Tests directory in Python path: {tests_path}")


def run_comprehensive_compliance_officer_tests():
    """
    Use pre-built comprehensive test suite to validate your Compliance Officer Agent.
    """
    print("🧪 Comprehensive Compliance Officer Testing")
    print("🚀 Running Compliance Officer test suite...")

    result = subprocess.run(
        ["python", "-m", "pytest", f"{tests_path}/test_compliance_officer.py", "-v", "--tb=short"],
        check=False,
    )

    if result.returncode == 0:
        print("✅ All Compliance Officer tests passed!")
    else:
        print("❌ Some tests failed. Check the output above for details.")


# Quick preview of available tests
try:
    from test_compliance_officer import TestComplianceOfficerAgent

    test_methods = [method for method in dir(TestComplianceOfficerAgent) if method.startswith("test_")]

    print("📝 Preview of Comprehensive Compliance Officer Tests:")
    for method_name in test_methods[:5]:
        method = getattr(TestComplianceOfficerAgent, method_name)
        doc = method.__doc__ or method_name.replace("_", " ").title()
        print(f"   • {doc}")
    if len(test_methods) > 5:
        print(f"   ... and {len(test_methods) - 5} more tests")
    print("\n💡 These tests validate regulatory compliance requirements!")
except Exception as e:
    print(f"ℹ️ Test suite will be available after implementing foundation_sar.py: {e}")

# Call the function
run_comprehensive_compliance_officer_tests()


📁 Tests directory in Python path: /Users/tarun/Documents/Tarun/Learning/Fintech & AI/Udacity/projects/agentic-ai-financial-compliance-sar/tests
📝 Preview of Comprehensive Compliance Officer Tests:
   • Test ComplianceOfficerAgent initializes properly
   • Test OpenAI API call uses correct parameters
   • LLM completeness_check=true does not waive mandated terminology / structure.
   • Test handling of empty LLM response
   • Test JSON extraction from code blocks
   ... and 7 more tests

💡 These tests validate regulatory compliance requirements!
🧪 Comprehensive Compliance Officer Testing
🚀 Running Compliance Officer test suite...
============================= test session starts ==============================
platform darwin -- Python 3.12.7, pytest-7.4.4, pluggy-1.0.0 -- /opt/anaconda3/bin/python
cachedir: .pytest_cache
rootdir: /Users/tarun/Documents/Tarun/Learning/Fintech & AI/Udacity/projects/agentic-ai-financial-compliance-sar
plugins: anyio-4.2.0
collecting ... collected 12 items


In [11]:
# COMPLETE AGENT TESTING - Two-Tier Approach
# Students: Use this to test both agents together

def complete_agent_testing_workflow():
    """
    Complete testing workflow using two-tier approach:
    
    TIER 1: Simple Smoke Tests (You write these)
    - Basic functionality verification
    - Quick sanity checks
    - Development debugging
    
    TIER 2: Comprehensive Test Suites (Pre-built for you)
    - Complex edge cases
    - Regulatory compliance validation
    - Professional-grade testing
    """
    print("🔬 Complete Agent Testing Workflow")
    print("=" * 50)
    
    print("\n📋 TIER 1: Simple Smoke Tests (DO FIRST)")
    print("   1. Write simple_risk_analyst_smoke_test() - verify basic functionality")
    print("   2. Write simple_compliance_officer_smoke_test() - verify basic functionality")
    print("   3. Fix any basic issues before moving to Tier 2")
    
    print("\n🧪 TIER 2: Comprehensive Test Suites (DO AFTER TIER 1 PASSES)")
    print("   1. Run comprehensive risk analyst test suite (10 comprehensive tests)")
    print("   2. Run comprehensive compliance officer test suite (10 comprehensive tests)")
    print("   3. Get detailed pass/fail results with specific feedback")
    
    print("\n💡 WHY THIS APPROACH?")
    print("   ✅ Tier 1: Quick feedback while developing")
    print("   ✅ Tier 2: Professional validation without writing complex tests")
    print("   ✅ Saves time: You focus on implementation, not test creation")
    print("   ✅ Better coverage: Our test suites test edge cases you might miss")

# Quick test runner when both agents are ready
def run_both_agents_quick_test():
    """Quick test of both agents using pre-built test suites"""
    print("🚀 Quick Test of Both Agents")
    print("📋 TODO: Uncomment when both agents are implemented")
    
    # Uncomment when ready:
    try:
        import pytest
        
        print("🔍 Running quick tests for both agents...")
        
        # Run a subset of tests for quick validation
        risk_result = pytest.main([
            f"{tests_path}/test_risk_analyst.py::TestRiskAnalystAgent::test_agent_initialization",
            f"{tests_path}/test_risk_analyst.py::TestRiskAnalystAgent::test_analyze_case_success",
            "-v"
        ])
        
        compliance_result = pytest.main([
            f"{tests_path}/test_compliance_officer.py::TestComplianceOfficerAgent::test_agent_initialization", 
            f"{tests_path}/test_compliance_officer.py::TestComplianceOfficerAgent::test_generate_compliance_narrative_success",
            "-v"
        ])
        
        if risk_result == 0 and compliance_result == 0:
            print("🎉 Both agents working! Ready for full test suite testing!")
        else:
            print("⚠️ Fix issues before running comprehensive tests")
            if risk_result != 0:
                print("   🔍 Risk Analyst needs fixes")
            if compliance_result != 0:
                print("   📝 Compliance Officer needs fixes")
                
    except ImportError as e:
        print(f"❌ Import Error: {e}")
        print("💡 Make sure both agents are implemented")

complete_agent_testing_workflow()
run_both_agents_quick_test()

🔬 Complete Agent Testing Workflow

📋 TIER 1: Simple Smoke Tests (DO FIRST)
   1. Write simple_risk_analyst_smoke_test() - verify basic functionality
   2. Write simple_compliance_officer_smoke_test() - verify basic functionality
   3. Fix any basic issues before moving to Tier 2

🧪 TIER 2: Comprehensive Test Suites (DO AFTER TIER 1 PASSES)
   1. Run comprehensive risk analyst test suite (10 comprehensive tests)
   2. Run comprehensive compliance officer test suite (10 comprehensive tests)
   3. Get detailed pass/fail results with specific feedback

💡 WHY THIS APPROACH?
   ✅ Tier 1: Quick feedback while developing
   ✅ Tier 2: Professional validation without writing complex tests
   ✅ Saves time: You focus on implementation, not test creation
   ✅ Better coverage: Our test suites test edge cases you might miss
🚀 Quick Test of Both Agents
📋 TODO: Uncomment when both agents are implemented
🔍 Running quick tests for both agents...
============================= test session starts =========

## 🔗 Phase 4 Preview: Agent Integration

Once both agents are working, you'll integrate them into a complete workflow.

In [12]:
# TODO: Preview of integrated workflow
# This will be fully implemented in the next notebook

def preview_integrated_workflow():
    """Preview of how agents will work together"""
    
    workflow_steps = [
        "1. 📊 Load and validate case data",
        "2. 🔍 Risk Analyst performs Chain-of-Thought analysis",
        "3. 👤 Human review and approval gate",
        "4. ✅ Compliance Officer generates ReACT narrative (if approved)",
        "5. 📄 Generate complete SAR document",
        "6. 📊 Log audit trail and efficiency metrics"
    ]
    
    print("🔗 Integrated SAR Processing Workflow:")
    for step in workflow_steps:
        print(step)
    
    print("\n💡 Key Benefits:")
    print("• Two-stage processing reduces AI costs")
    print("• Human oversight ensures regulatory compliance")
    print("• Complete audit trails for examination")
    print("• Standardized analytical approaches")

preview_integrated_workflow()

🔗 Integrated SAR Processing Workflow:
1. 📊 Load and validate case data
2. 🔍 Risk Analyst performs Chain-of-Thought analysis
3. 👤 Human review and approval gate
4. ✅ Compliance Officer generates ReACT narrative (if approved)
5. 📄 Generate complete SAR document
6. 📊 Log audit trail and efficiency metrics

💡 Key Benefits:
• Two-stage processing reduces AI costs
• Human oversight ensures regulatory compliance
• Complete audit trails for examination
• Standardized analytical approaches


## 📝 Development Checklist - Two-Tier Testing Approach

### ✅ Risk Analyst Agent (Phase 2)
- [ ] Implement Chain-of-Thought system prompt
- [ ] Create `analyze_case` method with error handling
- [ ] Add JSON parsing and validation
- [ ] **TIER 1**: Write simple smoke test (verify basic functionality)
- [ ] **TIER 2**: Run comprehensive pre-built test suite (10 comprehensive tests)
- [ ] Fix any issues identified by test suite

### ✅ Compliance Officer Agent (Phase 3)  
- [ ] Implement ReACT system prompt
- [ ] Create `generate_compliance_narrative` method
- [ ] Add narrative validation (word count, terminology)
- [ ] **TIER 1**: Write simple smoke test (verify basic functionality)
- [ ] **TIER 2**: Run comprehensive pre-built test suite (10 comprehensive tests)
- [ ] Fix any issues identified by test suite

### ✅ Testing Strategy Benefits
- [ ] **Time Savings**: Focus on implementation, not complex test creation
- [ ] **Better Coverage**: Pre-built test suites test edge cases you might miss
- [ ] **Quick Feedback**: Simple smoke tests for rapid development cycles
- [ ] **Professional Validation**: Comprehensive test suites ensure production readiness
- [ ] **Regulatory Compliance**: Built-in checks for SAR requirements

### 💡 **Testing Workflow**
1. **Start with Tier 1**: Write simple smoke tests to verify your agents don't crash
2. **Fix basic issues**: Iterate quickly with simple tests during development
3. **Move to Tier 2**: Run comprehensive test suites when basic functionality works
4. **Analyze results**: Use detailed feedback to improve agent performance
5. **Iterate**: Refine prompts and logic based on test results

## 🚀 Next Steps

1. **Complete Agent Implementation**: Finish both agent classes in the src/ directory
2. **Run Two-Tier Testing**: Start with smoke tests, then comprehensive test suites
3. **Workflow Integration**: Move to the next notebook for complete system integration
4. **Human-in-the-Loop**: Implement decision gates and review processes

## 📊 Available Test Suites Summary

**Risk Analyst Test Suite (10 tests):**
- Agent initialization and configuration
- Case analysis with valid JSON responses
- JSON parsing and error handling
- System prompt structure validation
- API call parameter verification
- Helper method functionality
- Edge case handling

**Compliance Officer Test Suite (10 tests):**
- Agent initialization and configuration
- Narrative generation with valid responses
- Word count validation (≤120 words)
- Regulatory citations inclusion
- JSON parsing and error handling
- ReACT prompt structure validation
- API call parameter verification

**Ready to build intelligent agents with professional-grade testing! 🕵️‍♀️**